# 08 - Fine-tune a small vision-language model on the plates (Qwen3.5-0.8B, free Colab T4)

In pillar 3bis, the big Groq VLM zero-shot missed subtle defects, and classic CV found them but couldn't name them. Question: can a **0.8B VLM fine-tuned on a few hundred labeled plates** beat the big zero-shot model?

- **Train set**: 500 new plates generated with the pillar 3bis generator (different random seed), clean images only
- **Test set**: the exact same 64 plates as `pillar3bis/data/plates` (same generator, same seed), so the numbers are comparable with `pillar3bis/02_vlm_inspection_eval.py`

**Before running**
1. `Runtime > Change runtime type > T4 GPU`
2. Upload `pillar3bis/data/make_plates.py` (folder icon on the left)
3. Optional: upload `pillar3bis/outputs/vlm_eval.json` to compare with the Groq VLM
4. `Runtime > Run all` (~25-35 min)

In [ ]:
%%capture
!pip install unsloth

## 1. Data: same test plates, new training plates

In [ ]:
import json, random
from pathlib import Path
import numpy as np
from PIL import Image

assert Path("make_plates.py").exists(), "Upload pillar3bis/data/make_plates.py first"
import make_plates as mp

# 1) test set: calling main() right after import reproduces pillar3bis/data/plates exactly (seed 7)
mp.main()
test_labels = json.loads((mp.OUT / "labels.json").read_text())
print(len(test_labels), "test plates in", mp.OUT)

# 2) train set: same generator, another seed, balanced classes, clean images only
mp.rng = np.random.default_rng(2024)
CLASSES = ["ok", "scratch", "dent", "stain", "missing_screw"]
train = []
for i in range(500):
    defect = CLASSES[i % len(CLASSES)]
    subtle = defect in ("scratch", "stain") and mp.rng.random() < 0.3
    img, _ = mp.make_plate(defect, subtle)
    train.append({"image": Image.fromarray(img[..., ::-1]), "defect": defect})  # BGR (opencv) -> RGB
random.Random(0).shuffle(train)
print(len(train), "training plates")
train[0]["image"]

## 2. Model and prompt

Same prompt as `pillar3bis/02_vlm_inspection_eval.py`. 16-bit LoRA (no QLoRA for Qwen3.5).

In [ ]:
import torch
from unsloth import FastVisionModel, is_bf16_supported

model, tokenizer = FastVisionModel.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    load_in_4bit=False,
    load_in_16bit=True,
    use_gradient_checkpointing="unsloth",
)

PROMPT = """You inspect brushed metal plates on a production line. A good plate has 4 screws (one in each
corner) and a clean, uniform surface. Possible defects: scratch, dent, stain, missing_screw.
Look at the whole surface carefully, defects can be small.
Return JSON: {"defect_type": "ok" | "scratch" | "dent" | "stain" | "missing_screw"}"""

def parse_defect(text):
    import re
    m = re.search(r'"defect_type"\s*:\s*"(\w+)"', text)
    return m.group(1) if m and m.group(1) in CLASSES else "invalid"

In [ ]:
@torch.no_grad()
def predict(image):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(image, prompt, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=30, do_sample=False, use_cache=True)
    return tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

def evaluate():
    FastVisionModel.for_inference(model)
    preds = {}
    for lab in test_labels:
        img = Image.open(mp.OUT / lab["file"]).convert("RGB")
        preds[lab["file"]] = parse_defect(predict(img))
    clean = [l for l in test_labels if not l["imaging_issue"]]
    acc = lambda subset: sum(preds[l["file"]] == l["defect"] for l in subset) / len(subset)
    escapes = sum(preds[l["file"]] == "ok" for l in clean if l["defect"] != "ok")
    print(f"accuracy all 64: {acc(test_labels):.0%} | clean images: {acc(clean):.0%} | "
          f"subtle: {acc([l for l in clean if l['subtle']]):.0%} | escapes on clean: {escapes}")
    return preds

print("BEFORE fine-tuning")
before = evaluate()

## 3. LoRA on both the vision encoder and the language model

`finetune_vision_layers=True`: the image encoder adapts to *our* images (brushed metal, thin scratches), not only the language side.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)
model.print_trainable_parameters()

def to_conversation(sample):
    answer = json.dumps({"defect_type": sample["defect"]})
    return {"messages": [
        {"role": "user", "content": [{"type": "text", "text": PROMPT}, {"type": "image", "image": sample["image"]}]},
        {"role": "assistant", "content": [{"type": "text", "text": answer}]},
    ]}

train_ds = [to_conversation(s) for s in train]

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=5,
        output_dir="outputs_vlm",
        report_to="none",
        seed=3407,
        # required for vision fine-tuning: the collator builds the batches itself
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,
    ),
)
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f} min, peak GPU memory {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

## 4. After fine-tuning, and comparison

In [ ]:
print("AFTER fine-tuning")
after = evaluate()

from collections import Counter
confusion = Counter((l["defect"], after[l["file"]]) for l in test_labels if not l["imaging_issue"])
print("\nclean images, truth -> prediction:")
for (t, p), n in sorted(confusion.items()):
    print(f"  {t:<14} -> {p:<14} {n}{'' if t == p else '   <-'}")

if Path("vlm_eval.json").exists():  # predictions of the big Groq VLM from pillar 3bis step 2
    groq = json.loads(Path("vlm_eval.json").read_text())
    truth = {l["file"]: l["defect"] for l in test_labels}
    for name, preds in groq.items():
        common = [f for f in preds if f in truth]
        print(f"\nGroq VLM ({name}), same images: {sum(preds[f] == truth[f] for f in common) / len(common):.0%} on {len(common)} images")

In [ ]:
model.save_pretrained("plates_lora")
tokenizer.save_pretrained("plates_lora")
json.dump({"before": before, "after": after}, open("plates_predictions.json", "w"), indent=1)
!zip -qr plates_finetune.zip plates_lora plates_predictions.json
from google.colab import files
files.download("plates_finetune.zip")

## What this shows (and what it doesn't)

- A tiny VLM trained on 500 labeled images usually becomes a much better *specialist* than a huge generalist used zero-shot. Same lesson as your CV career: in-domain labeled data wins.
- But it's still a classifier that only knows our 5 classes and our synthetic plates. On real images you'd need real labeled data, and the classic-CV golden-sample method from 3bis might still be the cheapest baseline to beat.
- Serving: exporting a fine-tuned VLM to Ollama/llama.cpp needs the vision part as a separate file (mmproj), so it's more involved than for text. For a vision API, `transformers` or vLLM behind FastAPI is the usual route (pillar 5 topic).
- Experiments: 100 vs 500 training images, `finetune_vision_layers=False`, adding degraded images with a `"retake"` label.